# 模型调用：结构化输出、工具协议与语义校验

区分模型生成、解析、业务校验和实际工具执行，写出能拒绝非法输入与结果的调用合同。

Python 3.10+ / CPU / 无密钥。固定教学 fixture，不调用模型或服务。先读对应正文，预测成功与失败，再运行并修改一个边界。

In [ ]:
from datetime import date
from decimal import Decimal

def validate_request(x):
    if not isinstance(x, dict) or set(x) != {'quantity', 'date', 'amount'}:
        raise ValueError('fields')
    if type(x['quantity']) is not int or not 1 <= x['quantity'] <= 100:
        raise ValueError('quantity')
    if not isinstance(x['date'], str): raise ValueError('date')
    date.fromisoformat(x['date'])
    if not isinstance(x['amount'], str): raise ValueError('amount')
    value = Decimal(x['amount'])
    if not value.is_finite() or value < 0: raise ValueError('amount')
    return {'quantity':x['quantity'], 'date':x['date'], 'amount':value}

base={'quantity':2, 'date':'2026-10-13', 'amount':'0.30'}
assert validate_request(base)['amount'] == Decimal('0.10')+Decimal('0.20')
for bad in [dict(base, quantity=True),dict(base, date='2026-02-30'),
            dict(base, amount='NaN'),dict(base, admin=True)]:
    try: validate_request(bad)
    except (ValueError, ArithmeticError): pass
    else: raise AssertionError('invalid request accepted')
checks={'strict_types':True,'calendar_validation':True,'finite_money':True,'unknown_fields':True}
print(checks)

## 记录与继续

在下一格 my_explanation 写下预测、改动与实际观察；运行后下载 mechanism_report.json，导回本篇笔记。下一次先独立解释边界，再对照旧记录。

In [ ]:
import json
from pathlib import Path
my_explanation = ''
report={'experiment':'lesson-llm-contract', 'scope':'stdlib fixtures; no model/service', 'checks':checks, 'my_explanation':my_explanation}
Path('mechanism_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
print(json.dumps(report,ensure_ascii=False,indent=2))
